In [2]:
import sympy as sp

In [3]:
# define independent variables 
x, t = sp.symbols('x t', real=True)

# define the rhs of the governing equation
u = sp.symbols('u')
c = sp.symbols('c', real=True)
nu = sp.symbols('nu', real=True, positive=True)
FU = -c * sp.Derivative(u, x) + nu * sp.Derivative(u, x, 2)
FU

-c*Derivative(u, x) + nu*Derivative(u, (x, 2))

In [4]:
# define q(t)
A = sp.Function('A')(t)
L = sp.Function('L')(t)
phi = sp.Function('phi')(t)

q = sp.Matrix([A, L, phi])

# define the ansatz u_hat(x; q)
ansatz = A * sp.sin(x/L + phi)

# compute partial derivatives du/dqi
du_dq = ansatz.diff(q).simplify()

du_dq

Matrix([
[                sin(x/L(t) + phi(t))],
[-x*A(t)*cos(x/L(t) + phi(t))/L(t)**2],
[           A(t)*cos(x/L(t) + phi(t))]])

In [5]:
# define the inner product according to the problem
def inner_prod_H(f, g):
    return sp.integrate(f*g,(x,0,2*sp.pi*L))

In [6]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    for j in range(n):
        M[i, j] = inner_prod_H(du_dq[i], du_dq[j]).simplify()

M

Matrix([
[                pi*L(t),                                              pi*A(t)*cos(2*phi(t))/2,                                  0],
[pi*A(t)*cos(2*phi(t))/2, pi*(6*pi*sin(2*phi(t)) + 3*cos(2*phi(t)) + 8*pi**2)*A(t)**2/(6*L(t)), -pi*(sin(2*phi(t))/2 + pi)*A(t)**2],
[                      0,                                   -pi*(sin(2*phi(t))/2 + pi)*A(t)**2,                    pi*A(t)**2*L(t)]])

In [7]:
# compute rhs from the ansatz
Fua = FU.subs(u,ansatz).doit()
Fua

-c*A(t)*cos(x/L(t) + phi(t))/L(t) - nu*A(t)*sin(x/L(t) + phi(t))/L(t)**2

In [8]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(du_dq[i], Fua).simplify()

f

Matrix([
[                                                              -pi*nu*A(t)/L(t)],
[pi*(c*L(t)*sin(2*phi(t)) + 2*pi*c*L(t) - nu*cos(2*phi(t)))*A(t)**2/(2*L(t)**2)],
[                                                                 -pi*c*A(t)**2]])

In [9]:
q_dot = M.inv()*f

q_dot.simplify()

In [10]:
q_dot

Matrix([
[-nu*A(t)/L(t)**2],
[               0],
[         -c/L(t)]])

# Prova altro